# Part 1.11 Evaluators

After this notebook you can score a workflow's output with code evaluators and an LLM
judge.

In [ ]:
from kea.config import MODEL, setup

setup("kea-claims-local")

## See it

An evaluator scores one output against the reference. Three of ours are plain code,
because the answer is checkable. One is an LLM judge, because "is this letter kind and
honest" is not. Run the finished target and all four on one example.

In [ ]:
from langsmith import Client

from kea.evals.claims import DATASET_NAME, EVALUATORS, build_dataset, make_target

client = Client()
if not client.has_dataset(dataset_name=DATASET_NAME):
    build_dataset(client)

examples = client.list_examples(dataset_name=DATASET_NAME, include_attachments=True)
example = next(e for e in examples if e.metadata["photo_id"] == "wheel-01")

output = make_target(MODEL)(example.inputs, example.attachments)
print("truth:", example.outputs)
for evaluator in EVALUATORS[:3]:
    print(evaluator(output, example.outputs))
print(EVALUATORS[3](output))

## Build it

Code evaluators take `outputs` and `reference_outputs` and return a key and a score.

In [ ]:
def damage_type_matches(outputs: dict, reference_outputs: dict) -> dict:
    return {
        "key": "damage_type_matches",
        "score": outputs["damage_type"] == reference_outputs["damage_type"],
    }


def severity_close(outputs: dict, reference_outputs: dict) -> dict:
    gap = abs(outputs["severity"] - reference_outputs["severity"])
    return {"key": "severity_close", "score": gap <= 1}


print(damage_type_matches(output, example.outputs))
print(severity_close(output, example.outputs))

The judge is a prompt, a model, and a feedback key. `openevals` wraps them into an
evaluator.

In [ ]:
from openevals.llm import create_llm_as_judge

from kea.claims import prompts
from kea.config import JUDGE_MODEL

print(prompts.LETTER_JUDGE)

judge = create_llm_as_judge(
    prompt=prompts.LETTER_JUDGE, model=JUDGE_MODEL, feedback_key="letter_quality"
)
judge(outputs={"payout_usd": output["payout_usd"], "letter": output["letter"]})

## Under the hood

- Make code evaluators **cheap and strict**, and use them wherever a rule can decide.
  Reach for a judge only for what a rule can't capture.
- The judge returns a score and its reasoning, so a failing letter explains itself in the
  UI.
- The judge model (`JUDGE_MODEL`) is separate from the model under test (`MODEL`), so
  swapping the model doesn't change who grades the answers.
- `kea.evals.claims.EVALUATORS` is the list the experiment in the next notebook uses.

Docs: [Evaluation concepts](https://docs.langchain.com/langsmith/evaluation-concepts),
[LLM-as-a-judge with the SDK](https://docs.langchain.com/langsmith/llm-as-judge-sdk).

## Recap

- An evaluator maps an output (and a reference) to a score.
- Use code where the answer is checkable and an LLM judge where it isn't.
- Keep the judge model separate from the model you are testing.

**Next:** [Part 1.12 Experiments](12_experiments.ipynb)